# 06: Querying Vector Embeddings

This notebook demonstrates how to query Neo4j using vector similarity search.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and Neo4j connection setup
- Completed [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb) to load vector embeddings into Neo4j

All environment detection, Neo4j connection, and configuration are handled in `00-import.ipynb`.

## Overview

This notebook demonstrates semantic search on your notes. Instead of searching for exact words, you can search by meaning - asking "find notes about brainstorming ideas" will return relevant notes even if they use different words. You can also combine this with graph patterns to get even more contextual results.

We'll:
1. Set up vector search using LangChain with LiteLLM proxy
2. Perform vector similarity search
3. Combine vector search with graph patterns for enhanced results


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores.neo4j_vector import Neo4jVector

print("✅ Additional libraries imported")


In [ ]:
# Settings, dependencies, driver, and neo4j_manager are already initialized in 00-import.ipynb
# They are available as: settings, dependencies, driver, neo4j_manager, IS_CONTAINER


## Set Up Vector Search

Create custom embeddings class that uses LiteLLM proxy and set up Neo4jVector.


In [ ]:
# Create custom embeddings class that uses LiteLLM proxy
class ProxyEmbeddings(OpenAIEmbeddings):
    """Custom embeddings class that uses LiteLLM proxy instead of direct OpenAI."""
    
    def __init__(self, proxy_base_url: str, api_key: str, model: str, **kwargs):
        api_url = f"{proxy_base_url}/v1"
        super().__init__(
            openai_api_base=api_url,
            openai_api_key=api_key,
            model=model,
            **kwargs
        )

# Create embeddings instance
proxy_base_url = f"http://{settings.litellm_proxy_host}:{settings.litellm_proxy_port}"
embedding_model = ProxyEmbeddings(
    proxy_base_url=proxy_base_url,
    api_key=settings.openai_api_key or "",
    model=settings.litellm_proxy_embedding_model,
)

print("✅ Embedding model configured")
# Proxy URL and model info already shown in 00-import.ipynb


In [ ]:
# Create Neo4jVector from existing index
kg_vector_search = Neo4jVector.from_existing_index(
    embedding=embedding_model,
    url=settings.neo4j_uri,
    username=settings.neo4j_username,
    password=settings.neo4j_password,
    database=settings.neo4j_database,
    index_name=settings.neo4j_vector_index_name
)

print("✅ Vector search configured")


## Basic Vector Search

Perform vector similarity search to find semantically similar notes.


In [ ]:
# Example search query
search_prompt = "project ideas and development tasks"

# Perform vector search
results = kg_vector_search.similarity_search(search_prompt, k=5)

# Display results
print(f"Found {len(results)} results for: '{search_prompt}'\n")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content[:200]}...")
    if hasattr(doc, 'metadata') and doc.metadata:
        print(f"   Metadata: {doc.metadata}\n")


## Enhanced Vector Search with Graph Context

Combine vector search with graph patterns for more contextual results.


In [ ]:
# Create personalized search with graph context
# This uses a retrieval_query to enhance vector search with graph patterns
kg_personalized_search = Neo4jVector.from_existing_index(
    embedding=embedding_model,
    url=settings.neo4j_uri,
    username=settings.neo4j_username,
    password=settings.neo4j_password,
    database=settings.neo4j_database,
    index_name=settings.neo4j_vector_index_name,
    retrieval_query="""
    WITH node AS note, score AS searchScore
    
    // Find entities in this note
    OPTIONAL MATCH (note)-[:CONTAINS]->(e:Entity)
    
    // Find other notes that share entities
    OPTIONAL MATCH (note)-[:CONTAINS]->(e:Entity)<-[:CONTAINS]-(related:Note)
    WHERE related <> note
    
    WITH note, searchScore, count(DISTINCT related) AS relatedNoteCount
    
    RETURN note.text AS text,
           searchScore,
           relatedNoteCount,
           {file_path: note.file_path, 
            file_name: note.file_name,
            related_notes: relatedNoteCount} AS metadata
    ORDER BY relatedNoteCount DESC, searchScore DESC
    LIMIT 10
    """
)

print("✅ Personalized search configured")


In [ ]:
# Perform personalized search
personalized_results = kg_personalized_search.similarity_search(search_prompt, k=5)

print(f"Personalized results for: '{search_prompt}'\n")
for i, doc in enumerate(personalized_results, 1):
    print(f"{i}. {doc.page_content[:200]}...")
    if hasattr(doc, 'metadata') and doc.metadata:
        print(f"   Related notes: {doc.metadata.get('related_notes', 0)}")
        print(f"   File: {doc.metadata.get('file_path', 'N/A')}\n")


## Next Steps

Proceed to:
- [**07-knn-graphrag.ipynb**](./07-knn-graphrag.ipynb): Use KNN and graph data science for advanced graph-powered RAG
